In [2]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split


In [ ]:
train_raw = pd.read_csv("agnews_train.csv")
test_raw = pd.read_csv("agnews_test.csv")

# 1. Define a cleaning function
def clean_agnews(df):
    df = df.rename(columns={
        "Class Index": "label",
        "Title": "title",
        "Description": "description"
    }).copy()

    df["title"] = df["title"].fillna("")
    df["description"] = df["description"].fillna("")

    df["text"] = df["title"].astype(str) + " " + df["description"].astype(str)

    df["text"] = df["text"].apply(lambda x: re.sub(r"\\+", " ", x))

    df["text"] = df["text"].apply(lambda x: re.sub(r"\s+", " ", x).strip())

    df["label"] = df["label"].astype(int) - 1
   
    df_clean = df[["text", "label"]].copy()

    return df_clean


# 2. Clean train and test separately

train_clean = clean_agnews(train_raw)
test_clean = clean_agnews(test_raw)

train_clean = (
    train_clean
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=5000, random_state=42))
    .reset_index(drop=True)
)


# 3. Split validation from train only

train_df, val_df = train_test_split(
    train_clean,
    test_size=0.1,   
    random_state=42,
    stratify=train_clean["label"]
)

# 4. Save final train / val / test

train_df.to_csv("agnews_clean_train.csv", index=False)
val_df.to_csv("agnews_clean_val.csv", index=False)
test_clean.to_csv("agnews_clean_test.csv", index=False)


/var/folders/fb/6ykw_w7x79l8l3z9_zfp6swr0000gn/T/ipykernel_57602/2866535532.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=5000, random_state=42))
